In [24]:
import nbformat

path = "/content/drive/MyDrive/Colab Notebooks/StyleBertVITS2_hirakawa.ipynb"

nb = nbformat.read(path, as_version=4)

# widgetsを完全リセット
nb.metadata["widgets"] = {
    "application/vnd.jupyter.widget-state+json": {
        "state": {},
        "version_major": 2,
        "version_minor": 0
    }
}

nbformat.write(nb, "/content/drive/MyDrive/Colab Notebooks/fixed_notebook_v2.ipynb")

In [26]:
import nbformat

path = "/content/drive/MyDrive/Colab Notebooks/StyleBertVITS2_hirakawa.ipynb"

nb = nbformat.read(path, as_version=4)

# metadataを最小構成にする
nb.metadata = {}

nbformat.write(nb, "/content/drive/MyDrive/Colab Notebooks/fixed_notebook_clean.ipynb")

In [1]:
# セル0: GPU確認
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024 // 1024} MiB')

Wed Apr  1 21:46:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# セル1: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# セル2: リポジトリクローン
import os
if not os.path.exists('/content/jp-natural-voice-app'):
    !git clone https://github.com/stangler/jp-natural-voice-app /content/jp-natural-voice-app
else:
    print('Already cloned')
os.chdir('/content/jp-natural-voice-app')

Already cloned


In [4]:
# セル3: パッケージセットアップ（固定バージョン）
!pip install torch==2.4.0+cu124 torchaudio==2.4.0+cu124 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install transformers==4.37.2 huggingface_hub==0.19.4 numpy==1.26.4 -q
!pip install -r requirements.txt -q
print('パッケージセットアップ完了 ✅')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.25 requires torchvision, which is not installed.
diffusers 0.37.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.19.4 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.19.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.19.4 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.3.0 requires huggingface-hub>=0.20.0, but you have huggingface-hub 0.19.4 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.3

In [5]:
# セル3.5: 【永続パッチ】jax/transformers 競合回避（セル3実行後は毎回実行）
filepath = '/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py'
with open(filepath, 'r') as f:
    lines = f.readlines()
new_lines = []
for line in lines:
    if 'import jax.numpy as jnp' in line:
        line = line.replace('import jax.numpy as jnp', 'pass  # jax disabled')
    new_lines.append(line)
with open(filepath, 'w') as f:
    f.writelines(new_lines)
print('jaxパッチ適用済み ✅')

jaxパッチ適用済み ✅


In [6]:
# セル4: bert_feature.py パッチ
# word2ph と text の長さ不一致による assert エラーを trim/pad で回避
patch_code = """
path = 'style_bert_vits2/nlp/japanese/bert_feature.py'
with open(path, 'r') as f:
    content = f.read()

old = '    text = text.replace(\"。\", \".\").replace(\"、\", \",\")\\n    assert len(word2ph) == len(text) + 2, text'

new = '''    text = text.replace(\"。\", \".\").replace(\"、\", \",\")
    expected = len(text) + 2
    if len(word2ph) != expected:
        if len(word2ph) > expected:
            word2ph = word2ph[:expected]
        else:
            word2ph = word2ph + [1] * (expected - len(word2ph))'''

count = content.count(old)
print(f'Found {count} occurrence(s)')
if count > 0:
    content = content.replace(old, new)
    with open(path, 'w') as f:
        f.write(content)
    print('bert_feature.py: Patched!')
else:
    print('bert_feature.py: Pattern not found (already patched or changed)')
"""

import subprocess
result = subprocess.run(['python3', '-c', patch_code], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

Found 0 occurrence(s)
bert_feature.py: Pattern not found (already patched or changed)



In [7]:
# セル5: lightning_fabric パッチ（毎回適用）
import importlib.util
if importlib.util.find_spec('lightning_fabric') is not None:
    import lightning_fabric
    lf_path = lightning_fabric.__file__.replace('__init__.py', 'utilities/cloud_io.py')
    with open(lf_path, 'r') as f:
        content = f.read()
    old = 'weights_only=True'
    new = 'weights_only=False'
    if old in content:
        content = content.replace(old, new)
        with open(lf_path, 'w') as f:
            f.write(content)
        print('lightning_fabric: Patched!')
    else:
        print('lightning_fabric: Already patched or not needed')
else:
    print('lightning_fabric not found - skip')

lightning_fabric: Already patched or not needed


In [8]:
# セル6: Drive からデータコピー
import os

# Drive の音声データ確認
DRIVE_WAV_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample'
wav_count = !find {DRIVE_WAV_PATH}/wavs -name '*.wav' 2>/dev/null | wc -l
esd_exists = os.path.exists(f'{DRIVE_WAV_PATH}/esd.list')
print(f'音声ファイル数: {wav_count[0]}')
print(f'esd.list: {esd_exists}')

# Data ディレクトリにコピー
os.makedirs('Data/hirakawa_sample/wavs', exist_ok=True)
!cp -n {DRIVE_WAV_PATH}/wavs/*.wav Data/hirakawa_sample/wavs/ 2>/dev/null || true
!cp -n {DRIVE_WAV_PATH}/esd.list Data/hirakawa_sample/esd.list 2>/dev/null || true

copied = !find Data/hirakawa_sample/wavs -name '*.wav' | wc -l
print(f'コピー済み wav: {copied[0]} ファイル ✅')

# Drive に前処理済みファイルがある場合はコピー（スキップ可）
DRIVE_BERT_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_preprocessed'
if os.path.exists(DRIVE_BERT_PATH):
    print('前処理済みファイルが Drive に見つかりました。コピーします...')
    !cp {DRIVE_BERT_PATH}/*.bert.pt Data/hirakawa_sample/wavs/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/*.npy Data/hirakawa_sample/wavs/ 2>/dev/null || true
    bert_count = !find Data/hirakawa_sample/ -name '*.bert.pt' | wc -l
    npy_count = !find Data/hirakawa_sample/ -name '*.npy' | wc -l
    print(f'.bert.pt: {bert_count[0]}, .npy: {npy_count[0]}')
else:
    print('前処理済みファイルなし → 次のセルで前処理を実行してください')

音声ファイル数: 188
esd.list: True
コピー済み wav: 188 ファイル ✅
前処理済みファイルなし → 次のセルで前処理を実行してください


In [9]:
# セル7: config.json 生成
import os
os.chdir('/content/jp-natural-voice-app')

!python preprocess_all.py \
    --model_name hirakawa_sample \
    --batch_size 4 \
    --epochs 100 \
    --use_jp_extra

print('config.json 生成確認:', os.path.exists('Data/hirakawa_sample/config.json'))

04-01 21:47:59 | DEBUG  | __init__.py:92 | try starting pyopenjtalk worker server
04-01 21:48:00 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
04-01 21:48:00 |  INFO  | train.py:72 | Step 1: start initialization...
model_name: hirakawa_sample, batch_size: 4, epochs: 100, save_every_steps: 1000, freeze_ZH_bert: False, freeze_JP_bert: False, freeze_EN_bert: False, freeze_style: False, freeze_decoder: False, use_jp_extra: True
04-01 21:48:00 | ERROR  | train.py:119 | Step 1: pretrained_jp_extra folder not found.
04-01 21:48:00 | DEBUG  | __init__.py:147 | pyopenjtalk worker server terminated
config.json 生成確認: True


In [10]:
# セル7.55: wavs/ をカレント基準で見せる（Audio not found 恒久対策）
import os, subprocess

os.chdir('/content/jp-natural-voice-app')

target = 'Data/hirakawa_sample/wavs'
link = 'wavs'

# 既存のwavsがディレクトリ/ファイルなら事故防止
if os.path.exists(link) and not os.path.islink(link):
    raise RuntimeError(f"'./{link}' exists but is not a symlink. Please remove or rename it.")

# symlink作成（既にあれば何もしない）
if not os.path.islink(link):
    os.symlink(target, link)

print("symlink:", link, "->", os.readlink(link))
print("head:", subprocess.check_output("ls wavs | head", shell=True, text=True))


symlink: wavs -> Data/hirakawa_sample/wavs
head: hirakawa-0.bert.pt
hirakawa-0.wav
hirakawa-0.wav.npy
hirakawa-100.bert.pt
hirakawa-100.wav
hirakawa-100.wav.npy
hirakawa-101.bert.pt
hirakawa-101.wav
hirakawa-101.wav.npy
hirakawa-102.bert.pt



In [11]:
# セル7.6: preprocess_text.py 実行（音素変換 → train.list / val.list 生成）
!python preprocess_text.py \
    --config-path Data/hirakawa_sample/config.json \
    --transcription-path Data/hirakawa_sample/esd.list \
    --cleaned-path Data/hirakawa_sample/esd.list.cleaned \
    --train-path Data/hirakawa_sample/train.list \
    --val-path Data/hirakawa_sample/val.list \
    --use_jp_extra

# 確認
with open('Data/hirakawa_sample/train.list', 'r', encoding='utf-8') as f:
    line = f.readline()
print('フィールド数:', len(line.strip().split('|')))
print('先頭1行:', line[:100])

04-01 21:48:11 | DEBUG  | __init__.py:92 | try starting pyopenjtalk worker server
04-01 21:48:12 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
100%|##########| 188/188 [00:01<00:00, 100.09it/s]
04-01 21:48:14 |  INFO  | preprocess_text.py:215 | Training set and validation set generation from texts is complete!
04-01 21:48:14 | DEBUG  | __init__.py:147 | pyopenjtalk worker server terminated
フィールド数: 7
先頭1行: wavs/hirakawa-0.wav|hirakawa_sample|JP|本日は青い森鉄道をご利用いただきましてありがとうございます.|_ h o N j i ts u w a a o i m o


In [12]:
# セル7.7: config.json 設定（batch_size / fp16 / save_every_steps）
import json

config_path = 'Data/hirakawa_sample/config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

config['train']['batch_size'] = 4
config['train']['fp16_run'] = True
config['train']['save_every_steps'] = 200

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"batch_size: {config['train']['batch_size']}")
print(f"fp16_run: {config['train']['fp16_run']}")
print(f"save_every_steps: {config['train']['save_every_steps']}")
print('config.json 更新完了 ✅')

batch_size: 4
fp16_run: True
save_every_steps: 200
config.json 更新完了 ✅


In [13]:
# セル7.9: JP BERTモデルを pytorch_model.bin 付きでローカル配置（bert_gen地雷回避）
import os
from transformers import AutoModel, AutoTokenizer

os.chdir('/content/jp-natural-voice-app')

MODEL_ID = "ku-nlp/deberta-v2-large-japanese-char-wwm"
LOCAL_DIR = "bert/deberta-v2-large-japanese-char-wwm"

os.makedirs(LOCAL_DIR, exist_ok=True)

print("Loading from Hub:", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
model = AutoModel.from_pretrained(MODEL_ID)

print("Saving with pytorch_model.bin (safe_serialization=False)")
tokenizer.save_pretrained(LOCAL_DIR)
model.save_pretrained(LOCAL_DIR, safe_serialization=False)

assert os.path.exists(os.path.join(LOCAL_DIR, "pytorch_model.bin"))
print("pytorch_model.bin generated successfully ✅")


Loading from Hub: ku-nlp/deberta-v2-large-japanese-char-wwm


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/520 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Saving with pytorch_model.bin (safe_serialization=False)
pytorch_model.bin generated successfully ✅


In [15]:
# セル8: 前処理（bert_gen / style_gen）
import subprocess

# bert_gen
bert_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True
).decode().strip())
print(f'既存の .bert.pt: {bert_count} ファイル')

if bert_count < 180:
    print('bert_gen.py を実行します...')
    !python bert_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ bert_gen 前処理済み → スキップ')

# style_gen（torchvision 競合回避のためアンインストール）
npy_count = int(subprocess.check_output(
    'find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True
).decode().strip())
print(f'既存の .npy: {npy_count} ファイル')

if npy_count < 180:
    print('style_gen.py を実行します...')
    !pip uninstall torchvision -y -q
    !python style_gen.py --config Data/hirakawa_sample/config.json
else:
    print('✅ style_gen 前処理済み → スキップ')

# 最終確認
bert_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.bert.pt" | wc -l', shell=True).decode().strip())
npy_count = int(subprocess.check_output('find Data/hirakawa_sample/ -name "*.npy" | wc -l', shell=True).decode().strip())
print(f'.bert.pt: {bert_count} ファイル')
print(f'.npy:     {npy_count} ファイル')

既存の .bert.pt: 188 ファイル
✅ bert_gen 前処理済み → スキップ
既存の .npy: 188 ファイル
✅ style_gen 前処理済み → スキップ
.bert.pt: 188 ファイル
.npy:     188 ファイル


In [ ]:
# セル9: 学習実行
import os
os.environ['MKL_THREADING_LAYER'] = 'GNU'

!python train_ms_jp_extra.py \
    -c Data/hirakawa_sample/config.json \
    -m Data/hirakawa_sample

In [ ]:
# セル10: モデルを Drive にバックアップ
import os

BACKUP_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_models'
os.makedirs(BACKUP_PATH, exist_ok=True)

!cp Data/hirakawa_sample/models/*.safetensors {BACKUP_PATH}/ 2>/dev/null || true
!cp Data/hirakawa_sample/config.json {BACKUP_PATH}/ 2>/dev/null || true

backed_up = os.listdir(BACKUP_PATH)
print(f'バックアップ完了: {backed_up} ✅')

In [ ]:
# セル11: 推論テスト
import subprocess

# 最新モデルを確認
models = !find Data/hirakawa_sample/models -name 'G_*.safetensors' | sort -V | tail -1
latest_model = models[0] if models else None
print(f'最新モデル: {latest_model}')

if latest_model:
    !python infer.py \
        --model_path {latest_model} \
        --config_path Data/hirakawa_sample/config.json \
        --text 'こんにちは、テストです。' \
        --output output_test.wav
    print('推論完了 ✅ → output_test.wav')
else:
    print('モデルが見つかりません。学習を完了してください。')